# Optimizing LLM Inference — A Case Study

Companion notebook for the [Optimizing LLM Inference lesson](https://ml-viz-ruby.vercel.app/courses/ml-in-practice/22-optimizing-llm-inference).

> **Tip:** use *File → Save a copy in Drive* so your edits persist.

**The idea in one sentence.** LLM serving has two phases with different physics — **prefill**
is compute-bound, **decode** is memory-bandwidth-bound — so before touching any knob you build
a first-principles cost model, find which constraint binds (TTFT, ITL, throughput, or cost),
and pull the lever that attacks *that* bottleneck. In this notebook we build that cost model
from scratch, verify its claims with asserts, simulate why continuous batching wins, and plot
the throughput-vs-latency frontier that every serving decision lives on.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
plt.style.use('dark_background')
rng = np.random.default_rng(42)

## 1 — From scratch: a first-principles inference cost model

A transformer forward pass over one token does roughly **2 FLOPs per parameter**
(one multiply + one add per weight). For a prompt of $s$ tokens, prefill does
$2 P s$ FLOPs *in parallel*; decode does $2P$ FLOPs *per generated token*, but must
re-read all $P \cdot \text{bytes}$ of weights from HBM each step.

The **roofline model** says achievable speed is the minimum of two ceilings:

$$\text{time} = \max\left(\frac{\text{FLOPs}}{\text{peak compute}},\; \frac{\text{bytes moved}}{\text{HBM bandwidth}}\right)$$

Whichever term wins tells you the phase's bottleneck. We model an H100-class GPU
(~990 TFLOP/s dense FP16, ~3.35 TB/s HBM) serving a 70B model, and add the KV-cache
sizing formula from the lesson:

$$\text{KV-cache bytes} = 2 \cdot L \cdot H_{kv} \cdot d \cdot s \cdot b \cdot \text{bytes per element}$$

In [ ]:
# ---- hardware (H100 SXM class) -------------------------------------------
PEAK_FLOPS = 990e12          # FP16 dense, FLOP/s
HBM_BW     = 3.35e12         # bytes/s

# ---- model (Llama-3-70B class) ---------------------------------------------
P_PARAMS   = 70e9            # parameters
N_LAYERS, N_KV_HEADS, HEAD_DIM = 80, 8, 128   # GQA: 8 KV heads

def weight_bytes(bytes_per_param=2.0):
    """Total bytes of model weights (FP16=2, FP8/INT8=1, INT4=0.5)."""
    return P_PARAMS * bytes_per_param

def prefill_seconds(s_tokens, bytes_per_param=2.0):
    """Roofline time to prefill a prompt of s tokens (single GPU, idealized)."""
    flops  = 2 * P_PARAMS * s_tokens
    bytes_ = weight_bytes(bytes_per_param)          # weights read once
    return max(flops / PEAK_FLOPS, bytes_ / HBM_BW)

def decode_ms_per_token(bytes_per_param=2.0, batch=1, kv_bytes_per_seq=0.0):
    """Roofline inter-token latency: one decode step reads all weights
    plus every active sequence's KV-cache."""
    flops  = 2 * P_PARAMS * batch
    bytes_ = weight_bytes(bytes_per_param) + batch * kv_bytes_per_seq
    return max(flops / PEAK_FLOPS, bytes_ / HBM_BW) * 1000

def kv_cache_gb(s_tokens, batch=1, bytes_per_el=2.0,
                L=N_LAYERS, H=N_KV_HEADS, d=HEAD_DIM):
    """KV-cache size: 2 (K and V) * L * H * d * s * b * bytes."""
    return 2 * L * H * d * s_tokens * batch * bytes_per_el / 1e9

# ---- the case-study workload: 8k-token RAG prompt, FP16 ---------------------
s = 8192
print(f"prefill of {s} tokens:      {prefill_seconds(s)*1000:8.0f} ms  (compute term wins)")
print(f"decode, batch=1, FP16:      {decode_ms_per_token(2.0):8.1f} ms/token  (bandwidth term wins)")
print(f"decode, batch=1, FP8:       {decode_ms_per_token(1.0):8.1f} ms/token")
print(f"KV-cache, one 8k sequence:  {kv_cache_gb(s):8.2f} GB")
print(f"KV-cache, batch of 32:      {kv_cache_gb(s, batch=32):8.1f} GB  (vs 140 GB of weights!)")

### Validate: decode is bandwidth-bound, prefill is compute-bound

Arithmetic intensity (FLOPs per byte read) tells you which side of the roofline
ridge each phase sits on. The ridge for our GPU is
$990\,\text{TFLOP/s} / 3.35\,\text{TB/s} \approx 295$ FLOPs/byte.
Decode at batch 1 does ~1 FLOP/byte (memory-bound, two orders of magnitude below
the ridge); prefill over an 8k prompt does ~8000 FLOPs/byte (compute-bound).
We also confirm the lesson's KV-cache number: one 8k sequence ≈ 2.7 GB.

In [ ]:
ridge = PEAK_FLOPS / HBM_BW                          # FLOPs/byte at the roofline ridge

intensity_decode  = (2 * P_PARAMS) / weight_bytes()  # batch=1: ~1 FLOP per byte
intensity_prefill = (2 * P_PARAMS * s) / weight_bytes()

assert intensity_decode < ridge / 100, 'decode sits deep in the memory-bound region'
assert intensity_prefill > ridge, 'prefill sits in the compute-bound region'

# decode latency floor: reading 140 GB of FP16 weights at 3.35 TB/s -> ~42 ms/token
floor_ms = decode_ms_per_token(2.0)
assert 35 < floor_ms < 55, 'FP16 decode floor is a few tens of ms per token'

# quantization moves ITL almost linearly with bytes per weight
assert abs(decode_ms_per_token(1.0) / floor_ms - 0.5) < 0.01, 'FP8 ~ halves ITL'

# KV-cache formula matches the lesson: 2*80*8*128*8192*1*2 bytes ~ 2.7 GB
assert abs(kv_cache_gb(8192) - 2.68) < 0.1

print(f'ridge {ridge:.0f} FLOPs/byte | decode intensity {intensity_decode:.1f} | '
      f'prefill intensity {intensity_prefill:.0f}')
print(f'decode floor {floor_ms:.1f} ms/token (FP16) -> {decode_ms_per_token(1.0):.1f} ms (FP8)')
print('all roofline + KV-cache checks passed')

## 2 — The library way: what the serving stack does with these facts

You never hand-roll this in production — serving stacks package each insight as a flag.
Our cost model maps directly onto them:

| Fact from our model | Production lever | vLLM flag / stack feature |
|---|---|---|
| Decode reads all weights per token | weight quantization | `--quantization fp8` / GPTQ, AWQ checkpoints |
| KV-cache grows per sequence, limits batch | paged allocation + KV quantization | PagedAttention (default), `--kv-cache-dtype fp8` |
| Shared prompt prefixes re-prefill for free | prefix caching | `--enable-prefix-caching` |
| Long prefills stall other requests' decode | chunked prefill | `--enable-chunked-prefill` |
| Idle batch slots waste the weight read | continuous batching | default scheduler (`max_num_seqs`) |

Sanity check against published numbers: our FP16 floor (~42 ms/token → ~24 tokens/s at
batch 1) matches reported single-request vLLM throughput for 70B-class models on one H100
within ~20% — a first-principles model gets you remarkably close.

The one lever worth *simulating* to believe is **continuous batching**. Below, the same
200 requests (output lengths 50–500 tokens) run through a 16-slot server twice:
**static batching** waits for the whole batch to finish before admitting new requests;
**continuous batching** refills each slot the moment its request completes.

In [ ]:
n_requests, batch_slots = 200, 16
out_lens = rng.integers(50, 501, n_requests)      # output tokens per request

# --- static batching: dispatch groups of 16, each group runs at its longest ---
static_steps = 0
for i in range(0, n_requests, batch_slots):
    static_steps += out_lens[i:i + batch_slots].max()

# --- continuous batching: a slot refills from the queue the moment it frees ---
queue = list(out_lens)
slots = [queue.pop(0) for _ in range(batch_slots)]
continuous_steps = 0
while slots:
    continuous_steps += 1
    slots = [r - 1 for r in slots if r > 1]        # every active slot decodes 1 token
    while queue and len(slots) < batch_slots:
        slots.append(queue.pop(0))

useful_tokens = out_lens.sum()
util_static     = useful_tokens / (static_steps * batch_slots)
util_continuous = useful_tokens / (continuous_steps * batch_slots)

print(f'static:     {static_steps:5d} steps | slot utilization {util_static:5.1%}')
print(f'continuous: {continuous_steps:5d} steps | slot utilization {util_continuous:5.1%}')
print(f'throughput ratio: {static_steps / continuous_steps:.2f}x in favour of continuous')

### Validate: continuous batching wins by keeping every slot busy

A static batch runs at the pace of its *longest* member — the short requests finish
early and their slots idle. Continuous batching refills those slots immediately, so
utilization stays near 100% until the queue drains. The gap grows with output-length
variance, which is exactly what chat traffic has.

In [ ]:
assert continuous_steps < static_steps
assert static_steps / continuous_steps > 1.3, 'continuous batching is a >1.3x win here'
assert util_continuous > util_static
print(f'continuous batching verified: {static_steps / continuous_steps:.2f}x '
      f'fewer decode steps for the same tokens')

## 3 — Visualize it: the throughput–latency frontier

Batching raises throughput *and* ITL together — one decode step now reads the weights
once but also every sequence's KV-cache, and the step serves the whole batch. Sweeping
batch size traces a **frontier**; quantization doesn't walk along the frontier, it
**shifts the whole curve**. The latency SLO then picks your operating point.

In [ ]:
kv_seq = kv_cache_gb(8192) * 1e9          # bytes of KV-cache per 8k sequence (FP16)
batches = np.arange(1, 129)

fig, ax = plt.subplots(figsize=(8, 5))
for bpp, label, color in [(2.0, 'FP16', '#f87171'),
                          (1.0, 'FP8 / INT8', '#facc15'),
                          (0.5, 'INT4', '#4ade80')]:
    kv = kv_seq * (bpp / 2.0)             # assume KV quantized alongside weights
    itl = np.array([decode_ms_per_token(bpp, b, kv) for b in batches])
    throughput = batches / (itl / 1000)   # tokens/s across the whole batch
    ax.plot(itl, throughput, color=color, lw=2, label=label)

ax.axvline(50, color='#818cf8', ls='--', lw=1.5, label='ITL SLO: 50 ms')
ax.set_xlabel('inter-token latency (ms/token)')
ax.set_ylabel('throughput (tokens/s, whole GPU)')
ax.set_title('Throughput vs ITL frontier — batch size sweeps each curve')
ax.set_xlim(0, 200)
ax.legend()
plt.tight_layout()
plt.show()

**What to notice.**

- Each curve is one precision; moving right along it = raising batch size. You buy
  throughput by spending per-request latency — the fundamental serving trade.
- **Quantization shifts the entire frontier up-and-left**: at the same 50 ms SLO
  (dashed line), FP8 serves roughly twice the tokens/s of FP16, and INT4 twice again.
  That is the honest way to state "quantization = 2× speedup" — more goodput *at a
  fixed SLO*, not just a faster demo at batch 1.
- The curves flatten as the KV-cache term starts to rival the weight term — at large
  batches the cache, not the weights, is what you're reading. This is why KV-cache
  quantization and GQA/MLA architectures matter at high concurrency.

## 4 — Tradeoffs: the lever table

| Lever | Typical win | Quality risk | Effort | Reach for it when |
|---|---|---|---|---|
| Current stack + continuous batching + paged KV | 2–10× throughput | none | config | always, first |
| Prefix / prompt caching | up to ~60–90% of prefill, big TTFT cut | none | one flag | shared prompts (RAG, multi-turn, agents) |
| Chunked prefill | kills ITL p95 spikes | none | one flag | long prompts mixed with chat |
| Consolidation + autoscaling | cost ∝ occupancy gain | none | infra | occupancy < ~40% |
| FP8 / INT8 weights (+ KV) | ~2× ITL & concurrency | low | recalibrate + evals | decode-bound, FP16 baseline |
| INT4 (GPTQ/AWQ) | ~3.5–4× vs FP16 | moderate | calibrate + per-slice evals | cost-bound, floor holds |
| Speculative decoding | 2–3× effective decode | none (verified) | draft model to run | templated/predictable outputs |
| Distillation / cascade routing | up to ~8× cost on routed share | needs evals | training project | narrow domain, stable traffic |
| Hardware swap (roofline-matched) | 1.3–2× | none | procurement | after software levers are exhausted |

**Common failure modes:** optimizing mean latency while p95 burns (batch coupling);
reporting throughput instead of goodput; quantizing before checking occupancy (model
risk taken to fix an infra problem); benchmarking at batch 1 and extrapolating; skipping
the eval suite because "INT8 is basically lossless" (it usually is — *per slice* is the check).

## ✏️ Your turn

**Exercise.** Implement `cost_per_million_output_tokens(gpu_hourly, batch_size, itl_ms)`:
given the GPU's hourly price, the number of concurrent sequences, and the inter-token
latency, return the serving cost of one million *output* tokens in dollars.

Steps: tokens generated per second = `batch_size * (1000 / itl_ms)`; scale to an hour;
cost per million = hourly price ÷ tokens per hour × 1e6. This one formula is how every
lever in this lesson turns into dollars — halve ITL (quantization) or double the batch
at fixed ITL (caching frees HBM) and the cost halves.

In [ ]:
def cost_per_million_output_tokens(gpu_hourly, batch_size, itl_ms):
    """
    gpu_hourly: GPU price in $/hour
    batch_size: concurrent sequences decoding
    itl_ms:     inter-token latency in milliseconds
    returns:    $ per 1,000,000 output tokens
    """
    # TODO(you): tokens per second across the batch
    tokens_per_sec = ...
    # TODO(you): tokens per hour, then dollars per million tokens
    return ...

print(cost_per_million_output_tokens(4.0, 32, 40.0))

In [ ]:
# Assertion — passes silently when your implementation is correct
got = cost_per_million_output_tokens(4.0, 32, 40.0)
assert abs(got - 4.0 / (32 * 25 * 3600) * 1e6) < 1e-6, f'expected ~1.389, got {got}'
assert abs(cost_per_million_output_tokens(4.0, 64, 40.0) - got / 2) < 1e-9, \
    'doubling batch at fixed ITL halves cost'
assert abs(cost_per_million_output_tokens(4.0, 32, 20.0) - got / 2) < 1e-9, \
    'halving ITL (e.g. FP8 quantization) halves cost'
print('all checks passed — you can now price any lever in $/Mtok')

<details><summary>Solution</summary>

```python
def cost_per_million_output_tokens(gpu_hourly, batch_size, itl_ms):
    tokens_per_sec = batch_size * (1000.0 / itl_ms)
    tokens_per_hour = tokens_per_sec * 3600
    return gpu_hourly / tokens_per_hour * 1e6
```

At $4/hr, batch 32, 40 ms ITL → 2.88M tokens/hour → **$1.39 per million output tokens**.
Quantize to FP8 (ITL → 20 ms) and it drops to $0.69; use the freed HBM to double the
batch and it halves again. Levers compound multiplicatively.
</details>

## Key takeaways

- **Two phases, two bottlenecks (verified):** prefill is compute-bound
  (~8000 FLOPs/byte, above the ~295 ridge), decode is bandwidth-bound (~1 FLOP/byte) —
  the FP16 decode floor for a 70B on one H100 is ~42 ms/token no matter the FLOPs.
- **Quantization moves ITL linearly with bytes per weight (verified):** FP8 halves it —
  and shifts the whole throughput-latency frontier, doubling goodput at a fixed SLO.
- **Continuous batching wins by refilling slots (verified):** same requests, same
  hardware, ~1.5–2× fewer decode steps than static batching on variable-length traffic.
- **The KV-cache is the concurrency ceiling:** one 8k sequence ≈ 2.7 GB (verified);
  at batch 32 the cache read rivals the weight read — hence KV quantization and GQA/MLA.
- **Every lever prices out through one formula:** $/Mtok = hourly ÷ (batch × 1000/ITL × 3600) × 10⁶ —
  which is why occupancy and quantization show up directly on the bill.

**Next:** [the full lesson & decision tree](https://ml-viz-ruby.vercel.app/courses/ml-in-practice/22-optimizing-llm-inference) ·
[Inference Optimization & Serving](https://ml-viz-ruby.vercel.app/courses/ml-in-practice/12-inference-optimization-and-serving) ·
[GPUs for Deep Learning](https://ml-viz-ruby.vercel.app/courses/gpu-programming/04-gpus-for-deep-learning)